# Lab 3: Hashes to ashes

As usual, start by writing <b style="color:red">HACKOOLIQUES</b> here.

## A) Pearson hashing

You are given a permutation of the set of 2-byte strings as a Python dictionary:

In [1]:
import pickle

with open('perm', 'rb') as file:
    perm = pickle.load(file)

If you want to know the image of, say, `3c2d` under this permutation, just look at the corresponding entry: 

In [9]:
perm[bytes.fromhex("0002")].hex()
perm[bytes.fromhex("3c2d")].hex()

'bf21'

Use this permutation to construct a Pearson hash function turning arbitrarily long byte arrays into 2-byte hashes (you may assume the input contains an even number of bytes to avoid padding issues).

Make sure that you get the following values:

$m = \verb|b"hello!"|$, $h = \tt{1b3f}$

$m = \verb|b"HELLO?"|$, $h = \tt{fea5}$

$m = \verb|b"A longer one"|$, $h = \tt{e1ba}$

In [21]:
def pearson(B, perm):
    h = b'\x00\x00'
    for i in range(0, len(B), 2):
        pair = B[i:i+2]
        h = perm[bytes(a ^ b for a, b in zip(h, pair))]
    return h.hex()
M = [b"hello!",b"HELLO?",b"A longer one"]
[print(pearson(m, perm)) for m in M]


1b3f
fea5
e1ba


[None, None, None]

## B) Birthday attack

From now on, you play the attacker and try to break the above (insecure) hash function, to which you have unlimited access (*i.e.*, you can compute as many hashes as you want with it).

How many hashes of randomly generated strings do you _expect_ it would take before a collision is found for this hash function?

2 octets = 16 bits -> 2^16 = N = taille de l’espace de clés = 4 294 967 296

- p≈1−exp(−k(k−1)/2N​)
- k≈sqrt(2Nln(1/(1−p)​))

| Probabilité (p) | Nombre d’essais (k) (≈) |
| --------------: |  ----------------------: |
|             25% |               **195** |
|             50% |               **302** |
|             75% |              **427** |
|             99% |              **777** |


Check how lucky you are today by performing a birthday attack on the Pearson hash function. How many hashes did you actually need to compute? (compare with the expected value and discuss)

In [59]:
"""
Recherche de collision (birthday attack) sur une version étendue de Pearson
qui produit un hash de 'out_bytes' octets (ici par défaut 4 octets = 32 bits).

Usage: python3 pearson_bday.py
"""

import os
import random
import time
from typing import List

# Exemple : permutation de 256 octets. Remplace par ta perm si besoin.
# Doit être une liste de 256 entiers 0..255 (permutation).
perm = list(range(256))
random.seed(0)
random.shuffle(perm)

def pearson_extended(data: bytes, perm: List[int], out_bytes: int = 4) -> bytes:
    """
    Version étendue de Pearson : calcule out_bytes octets de hash.
    Ici chaque octet de sortie est obtenu en appliquant l'algorithme Pearson classique
    avec un état initial différent (par exemple 0..out_bytes-1).
    """
    out = bytearray(out_bytes)
    for i in range(out_bytes):
        h = i & 0xFF  # état initial distinct pour chaque octet de sortie
        for b in data:
            h = perm[h ^ b]
        out[i] = h
    return bytes(out)

def random_message(min_len=6, max_len=32) -> bytes:
    """Génère un message aléatoire de longueur entre min_len et max_len."""
    L = random.randint(min_len, max_len)
    return os.urandom(L)

def find_collision(perm: List[int],
                   out_bytes: int = 2,
                   max_attempts: int = 10_000_000,
                   report_every: int = 50_000):
    """
    Cherche une collision en générant des messages aléatoires.
    Retourne (hash, msg1, msg2, attempts, elapsed_seconds) quand collision trouvée.
    """
    seen = {}  # map hash_bytes -> message
    attempts = 0
    t0 = time.time()
    while attempts < max_attempts:
        attempts += 1
        msg = random_message()
        h = pearson_extended(msg, perm, out_bytes=out_bytes)
        if h in seen:
            if seen[h] != msg:
                elapsed = time.time() - t0
                return h, seen[h], msg, attempts, elapsed
            # sinon : même message réapparu, on continue
        else:
            seen[h] = msg

        if attempts % report_every == 0:
            now = time.time()
            print(f"[{attempts:,}] essais — taille table: {len(seen):,} — "
                  f"elapsed {now - t0:.1f}s")

    return None  # pas trouvé dans le budget d'essais

# paramètres
OUT_BYTES = 2            # 4 octets = 32 bits
MAX_ATTEMPTS = 2_000_000 # limite pour éviter runtime infini (ajuste si tu veux)
REPORT_EVERY = 50_000

def main():
    # print("Démarrage recherche collision Pearson ({} octets de sortie)…".format(OUT_BYTES))
    result = find_collision(perm, out_bytes=OUT_BYTES,
                            max_attempts=MAX_ATTEMPTS,
                            report_every=REPORT_EVERY)
    if result is None:
        print(f"Aucune collision trouvée après {MAX_ATTEMPTS:,} essais.")
    else:
        h, m1, m2, attempts, elapsed = result
        # print("\n--- Collision trouvée ! ---")
        # print(f"Hash (hex) : {h.hex()}")
        # print(f"Tentatives : {attempts:,}")
        # print(f"Temps écoulé : {elapsed:.2f}s")
        # print(f"Message 1 (len {len(m1)}): {m1!r}")
        # print(f"Message 2 (len {len(m2)}): {m2!r}")
        return attempts
    
attempts = [main() for _ in range(10000)]
import statistics as s
print(s.mean(attempts), s.median(attempts), s.stdev(attempts))


319.213 302.0 164.64014476647813


Sur 10k tentatives : moyenne 319.213, mediane 302

## C) Extension attack

The situation is much worse than that! since the Pearson hash function is not designed to be cryptographically secure. Convince yourself that, given two strings, such as $a = \verb|"target"|$ and $b = \verb|"beginning of the end"|$, it is always possible to append two characters at the end of $b$ to get a new string $b'$ such that $H(b') = H(a)$. Be fair game by using only information known to the attacker (<i>i.e.</i>, it is forbidden to look at $\tt{perm}$ but you can perform as many hash evaluations as you like).

In [ ]:
def pearson(B, perm):
    h = b'\x00\x00'
    for i in range(0, len(B), 2):
        pair = B[i:i+2]
        h = perm[bytes(a ^ b for a, b in zip(h, pair))]
    return h


a=b"target"
p_a = pearson(a, perm)
b=b"beginning of the end"
p_b = pearson(b, perm)
print(a, p_a.hex())
print(b, p_b.hex())


 

b'target' 9902
b'beginning of the end' fe1c
Found: 626567696e6e696e67206f662074686520656e64bc81


In [8]:
for i in range(256):
    for j in range(256):
        trial = b + bytes([i, j])
        if pearson(trial, perm) == p_a:
            print("Found:", trial)
            break   

Found: b'beginning of the end\xbc\x81'


Complexite 2^8 * 2^8 = 2^16

What's the bit complexity of your attack? Try to make it as efficient as possible.

[ _Hint_: it's easy to make it in $2^{16}$ steps, but with a bit of cleverness you can take it down to $2^0$... ]

In [15]:
bb =bytes([a^b for a,b in zip(p_b,bytes.fromhex("FFFF"))])
print(pearson( b+bb, perm).hex())

4759


## D) Dictionary/brute-force attack

Now a "real-world" (broken) example. Some hackers have leaked a large database of hashed password from a well-known online retailer, where we can read hashed passwords:

```
Alice  753692ec36adb4c794c973945eb2a99c1649703ea6f76bf259abb4fb838e013e
Bob    751bc918ef403f5b898e321b6266b3e7f8fc8decfcc9a83bb83ce771527a828d
Carol  f7092ca05af91105bfdfb64b29196420f936b2f01dd91d62355c549e44312f26
David  e67ad6e81a23071f87a7edb3b45752d4b54e3d41d2b23d05038db23b1c2802fd  
...
```

Noticing that they are 64 hexadecimal digits long and that the value next to Alice's name is the SHA-256 of `Hallo` (you should be able to check that!), you suspect that these are just unsalted SHA-256 hashes. 

You think Bob might be the kind of person to use his birthdate as password, and you know he was born after 2000. Can you recover his password? What's the bit complexity of your attack?

In [28]:
import hashlib

Alice="753692ec36adb4c794c973945eb2a99c1649703ea6f76bf259abb4fb838e013e"
Bob="751bc918ef403f5b898e321b6266b3e7f8fc8decfcc9a83bb83ce771527a828d"
Carol="f7092ca05af91105bfdfb64b29196420f936b2f01dd91d62355c549e44312f26"
David="e67ad6e81a23071f87a7edb3b45752d4b54e3d41d2b23d05038db23b1c2802fd"

def sha256(m):
    return hashlib.sha256(m.encode()).digest().hex()

print("Hallo == sha256(Alice):", sha256("Hallo") == Alice)

for y in range(2000,2026):
    for m in range(1,13):
        for d in range(1,32):
            date_str = f'{m:02}/{d:02}/{y:04}'
            if sha256(date_str) == Bob : 
                print("Bob:", date_str)
                break

Hallo == sha256(Alice): True
Bob: 03/25/2008


[ Bonus points if you recover Carol's secret password as well! ]

In [30]:
with open("../rockyou.txt.sha256.txt") as f:
    for line in f.readlines():
        hash,pwd = line.split(":",1)
        if hash == Carol:
            print("Carol:", pwd)
            break

Carol: 708xy95BobaFett



Attaque par rainbow table
(18.5) s Carol: 708xy95BobaFett

[ Extra bonus points if you recover David's secret password as well! Good luck though... ]

En l'absence de social engineering ou d'entrée dans la rainbow table, "impossible" de retrouver le pwd 